In [ ]:
# Nạp định nghĩa từ 03 mà không chạy train — cùng cách 01/01_1/06 vẫn làm.
from pathlib import Path

_cwd = Path.cwd().resolve()
_ROOT = next((p for p in (_cwd, *_cwd.parents)
              if (p / "Code" / "03_train_model.ipynb").is_file()), None)
assert _ROOT is not None, f"Không thấy Code/03_train_model.ipynb quanh {_cwd}"
_CODE = _ROOT / "Code"

if not globals().get("SDC_DEFS_LOADED"):
    SDC_IMPORT_ONLY = True
    try:
        get_ipython().run_line_magic("run", f'-i "{_CODE / "03_train_model.ipynb"}"')
    finally:
        del SDC_IMPORT_ONLY

import pandas as pd
from IPython.display import display

try:
    import scapy
    print("scapy  :", scapy.VERSION)
except ImportError:
    raise SystemExit("Cần scapy để đọc pcap: pip install scapy")


## 1. Liệt kê pcap mới

Bỏ qua file rác AppleDouble (`._*.pcap`), cùng quy tắc với `01_feature_extract.ipynb`.

In [ ]:
from scapy.all import PcapReader, Ether

PCAP_DIR = ROOT / "Data" / "data_pcap"
OUT_DIR = FEAT_DIR  # Data/features, đã định nghĩa ở 03
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert PCAP_DIR.exists(), f"Không tìm thấy {PCAP_DIR}"

pcap_files = sorted(p for p in PCAP_DIR.glob("*.pcap") if not p.name.startswith("._"))
print(f"{len(pcap_files)} file pcap")
for p in pcap_files:
    print(f"  {p.name}  ({p.stat().st_size / 1e6:.1f} MB)")


## 1bis. Bang ten thiet bi (hostname) theo MAC

`Data/data_pcap/device_map.csv` (hostname, mac, ip) lay tu bang DHCP lease +
ARP cua router luc bat goi (`host_name_join.png`, `MAC.png`). Dung de gan them
cot `device` de doc ben canh `mac` tren moi output - khong thay `mac` vi do
moi la khoa session that. MAC la (khong co trong bang) duoc gan nhan
`Unknown-<mac>` thay vi bo trong.

Khi co danh sach thiet bi moi, chi can sua lai `device_map.csv` roi chay lai
notebook - khong can sua code.


In [ ]:
import csv

DEVICE_MAP_CSV = PCAP_DIR / "device_map.csv"


def load_device_map(path):
    mapping = {}
    if not path.exists():
        return mapping
    with open(path, newline="", encoding="utf-8-sig") as f:
        for row in csv.DictReader(f):
            mac = normalize_mac(row["mac"])
            if mac:
                mapping[mac] = row["hostname"].strip()
    return mapping


mac_to_device = load_device_map(DEVICE_MAP_CSV)


def device_for_mac(mac):
    if mac is None:
        return "Unknown"
    return mac_to_device.get(mac, f"Unknown-{mac.replace(':', '')}")


print(f"{len(mac_to_device)} thiet bi trong {DEVICE_MAP_CSV.name}")


## 2. Trích xuất theo MAC nguồn

Mỗi file pcap = một lần bắt gói trên cả mạng, nhiều thiết bị trộn chung. Không có
`device_mac` biết trước như `POWER`/IoT Sentinel, nên xử lý toàn bộ gói trong file
(không lọc theo `packet_from_device`) và tự gom theo MAC:

- **DHCP** — gắn theo `client_mac` (`BOOTP.chaddr`, qua `extract_dhcp`) nếu có, để cả gói
  OFFER/ACK do server gửi vẫn quy đúng về thiết bị xin cấp IP; nếu thiếu thì rơi về
  `Ether.src` của chính gói đó.
- **DNS/mDNS/TLS** — luôn do chính thiết bị phát ra (`extract_dns` chỉ lấy query,
  ClientHello luôn từ client), nên gắn thẳng theo `Ether.src`.

`(mac, capture_file)` là đơn vị session — giữ nguyên record thô song song (không chỉ
bảng phẳng) để mục 4 gọi `aggregate()` mà không phải suy ngược từ CSV.

In [ ]:
from collections import defaultdict

from tqdm.auto import tqdm

dhcp_rows, dns_rows, mdns_rows, tls_rows = [], [], [], []
summary_counts = defaultdict(lambda: {"n_dhcp": 0, "n_dns": 0, "n_mdns": 0, "n_tls": 0})
records_by_session = defaultdict(list)  # (mac, capture_file) -> [record, ...] cho aggregate()

for pcap_path in tqdm(pcap_files, desc="capture files"):
    tls_streams = {}  # mac -> TLSStreamReassembler, tách riêng theo thiết bị

    with PcapReader(str(pcap_path)) as reader:
        for pkt in reader:
            if not pkt.haslayer(Ether):
                continue
            ts = float(pkt.time)

            dhcp_row = extract_dhcp(pkt, ts)
            if dhcp_row:
                mac = dhcp_row.get("client_mac") or dhcp_row.get("src_mac")
                if mac:
                    row = {"mac": mac, "capture_file": pcap_path.name, **dhcp_row}
                    dhcp_rows.append(row)
                    records_by_session[(mac, pcap_path.name)].append(dhcp_row)
                    summary_counts[(mac, pcap_path.name)]["n_dhcp"] += 1

            mac = packet_source_mac(pkt)
            if mac is None:
                continue

            for dns_row in extract_dns(pkt, ts):
                row = {"mac": mac, "capture_file": pcap_path.name, **dns_row}
                records_by_session[(mac, pcap_path.name)].append(dns_row)
                if dns_row["is_mdns"]:
                    mdns_rows.append(row)
                    summary_counts[(mac, pcap_path.name)]["n_mdns"] += 1
                else:
                    dns_rows.append(row)
                    summary_counts[(mac, pcap_path.name)]["n_dns"] += 1

            tls_stream = tls_streams.setdefault(mac, TLSStreamReassembler())
            for tls_row in tls_stream.feed(pkt, ts):
                tls_rows.append({"mac": mac, "capture_file": pcap_path.name, **tls_row})
                records_by_session[(mac, pcap_path.name)].append(tls_row)
                summary_counts[(mac, pcap_path.name)]["n_tls"] += 1

dhcp_df = pd.DataFrame(dhcp_rows)
dns_df = pd.DataFrame(dns_rows)
mdns_df = pd.DataFrame(mdns_rows)
tls_df = pd.DataFrame(tls_rows)
summary_df = pd.DataFrame(
    [{"mac": mac, "capture_file": capture_file, **counts}
     for (mac, capture_file), counts in summary_counts.items()]
)

print("dhcp_df:", dhcp_df.shape, "dns_df:", dns_df.shape,
      "mdns_df:", mdns_df.shape, "tls_df:", tls_df.shape)
print(f"{summary_df[['mac']].drop_duplicates().shape[0]} MAC nguồn, "
      f"{len(records_by_session)} session (mac, capture_file)")


Gan cot `device` (hostname) cho tung dataframe theo `mac`.

In [ ]:
for _df in (dhcp_df, dns_df, mdns_df, tls_df, summary_df):
    if not _df.empty:
        _df.insert(_df.columns.get_loc("mac") + 1, "device", _df["mac"].map(device_for_mac))


## 3. Lưu output raw

Ghi đè mỗi lần chạy — thư mục `data_pcap/` còn đang nhận thêm file.

In [ ]:
dhcp_df.to_csv(OUT_DIR / "dhcp_features_capture.csv", index=False)
dns_df.to_csv(OUT_DIR / "dns_features_capture.csv", index=False)
mdns_df.to_csv(OUT_DIR / "mdns_features_capture.csv", index=False)
tls_df.to_csv(OUT_DIR / "tls_features_capture.csv", index=False)
summary_df.to_csv(OUT_DIR / "capture_summary_capture.csv", index=False)

print("Đã lưu vào", OUT_DIR.resolve())
for f in ["dhcp_features_capture.csv", "dns_features_capture.csv", "mdns_features_capture.csv",
          "tls_features_capture.csv", "capture_summary_capture.csv"]:
    print(" -", f)


## 4. Gộp thành bảng 40 cột (sẵn sàng cho model)

Gọi đúng `aggregate()` của `03_train_model.ipynb` trên record thô đã giữ ở mục 2 —
không tự gộp tay, tránh lệch với lúc train/serve (xem `README.md` mục 2.7).

In [ ]:
frame, feature_groups = load_sessions(SESSIONS_PATH)
features = feature_groups["all"]

agg_rows = []
for (mac, capture_file), records in records_by_session.items():
    row = {"mac": mac, "capture_file": capture_file, **aggregate(records, features)}
    agg_rows.append(row)

session_df = pd.DataFrame(agg_rows)

session_df.insert(1, "device", session_df["mac"].map(device_for_mac))
session_df.to_csv(OUT_DIR / "session_features_capture.csv", index=False)

print(f"{len(session_df)} session x {len(features)} cột feature")
print("Đã lưu", OUT_DIR / "session_features_capture.csv")
display(session_df[["mac", "device", "capture_file", "has_dhcp", "has_dns", "has_mdns", "has_tls",
                     "n_sources", "dhcp_vci", "tls_sni_tokens"]])


## 5. Kiểm tra nhanh (preview)

Số nguồn có mặt theo MAC, cùng tinh thần mục 7 của `01_feature_extract.ipynb`.

In [ ]:
print("Phân bố số nguồn feature có mặt / (mac, capture_file):")
display(session_df["n_sources"].value_counts().sort_index())

print()
print("Chi tiết theo MAC:")
display(session_df.groupby(["mac", "device"])[["has_dhcp", "has_dns", "has_mdns", "has_tls"]].max())
